# Alpha Connectome: train many GRUs from one data stream

Downloads the starter pack once, then trains every experiment in `experiments.json` side by side:
each batch of sequences is read once and used to train all models; evaluation is shared too.

**Runtime:** Runtime > Change runtime type > **A100 or L4 GPU** and **High-RAM**.
Results (checkpoints, logs, ONNX submissions) go to Google Drive if `USE_DRIVE = True`, so they survive a disconnect.

Code below is generated from `pipeline/` by `colab/build_notebook.py`; edit there, not here.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "CPUs: $(nproc)"; free -g | head -2; df -h /content | tail -1

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
CPUs: 12
               total        used        free      shared  buff/cache   available
Mem:             167           2         161           0           3         164
overlay         236G   43G  194G  18% /


In [2]:
USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs' if USE_DRIVE else '/content/runs'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
import os; os.makedirs(OUT_DIR, exist_ok=True); print('results ->', OUT_DIR)

Mounted at /content/drive
results -> /content/drive/MyDrive/wunder-connectome/runs


## 1. Data (33.7 GB, streamed straight into place; ~10-20 min, skipped if already there)

In [3]:
%%bash
mkdir -p /content/wnn && cd /content/wnn
if [ ! -f datasets/valid.parquet ]; then
  time (curl -L --retry 20 -sS https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --strip-components=1)
fi
ls -la datasets baseline && du -sh datasets

baseline:
total 1492
drwxr-xr-x 2  501 staff   4096 Oct 23 00:20 .
drwxr-xr-x 5 root root    4096 Oct 23 00:20 ..
-rw-r--r-- 1  501 staff    163 Dec 31  2019 ._baseline.onnx
-rw-r--r-- 1  501 staff 770975 Dec 31  2019 baseline.onnx
-rw-r--r-- 1  501 staff    163 Dec 31  2019 ._baseline_submission.zip
-rw-r--r-- 1  501 staff 716978 Dec 31  2019 baseline_submission.zip
-rw-r--r-- 1  501 staff    163 Dec 31  2019 ._README.md
-rw-r--r-- 1  501 staff    659 Dec 31  2019 README.md
-rw-r--r-- 1  501 staff    163 Dec 31  2019 ._solution.py
-rw-r--r-- 1  501 staff   2987 Dec 31  2019 solution.py

datasets:
total 34128648
drwxr-xr-x 2  501 staff        4096 Oct 23 00:26 .
drwxr-xr-x 5 root root         4096 Oct 23 00:20 ..
-rw-r--r-- 1  501 staff         163 Dec 31  2019 ._train.parquet
-rw-r--r-- 1  501 staff 29240219011 Dec 31  2019 train.parquet
-rw-r--r-- 1  501 staff         163 Dec 31  2019 ._valid_mask.parquet
-rw-r--r-- 1  501 staff    91972846 Dec 31  2019 valid_mask.parquet
-rw-r--r-- 

tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCH

In [4]:
!pip -q install onnx onnxruntime
%cd /content/wnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.3 MB/s eta 0:00:00
/content/wnn


## 2. Pipeline code

In [5]:
!mkdir -p /content/wnn/pipeline

In [6]:
%%writefile /content/wnn/pipeline/__init__.py

Writing /content/wnn/pipeline/__init__.py


In [7]:
%%writefile /content/wnn/pipeline/data.py
"""Fast sequence loading from the competition parquet files.

Each row group is one 20k-step sequence. Readers keep one ParquetFile per thread
(re-opening parses a large footer) and write straight into a preallocated batch array.
"""
import threading
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pyarrow.parquet as pq

from utils import FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS, WARMUP

N_FEATURES = len(FEATURE_COLUMNS)


class SeqStore:
    def __init__(self, path: str, n_threads: int = 4):
        self.path = path
        pf = pq.ParquetFile(path)
        self.n_seqs = pf.num_row_groups
        self.has_mask = "is_scored" in pf.schema_arrow.names
        self.cols = list(FEATURE_COLUMNS) + list(TARGET_COLUMNS) + (["is_scored"] if self.has_mask else [])
        self._local = threading.local()
        self._pool = ThreadPoolExecutor(n_threads)
        self._cache = None

    def enable_cache(self):
        """Keep every sequence in RAM after its first read (~9 MB each)."""
        self._cache = {}

    def _pf(self):
        if not hasattr(self._local, "pf"):
            self._local.pf = pq.ParquetFile(self.path)
        return self._local.pf

    def _load_into(self, i, g, X, Y, M):
        if self._cache is not None and g in self._cache:
            X[i], Y[i], M[i] = self._cache[g]
            return
        df = self._pf().read_row_group(int(g), columns=self.cols, use_threads=False).to_pandas()
        a = df.to_numpy(np.float32)
        X[i] = a[:, :N_FEATURES]
        Y[i] = a[:, N_FEATURES:N_FEATURES + 2]
        need = np.arange(SEQUENCE_LENGTH) >= WARMUP
        M[i] = need & (a[:, -1] > 0.5) if self.has_mask else need
        if self._cache is not None:
            self._cache[g] = (X[i].copy(), Y[i].copy(), M[i].copy())

    def load(self, ids):
        """-> X [B,T,112] f32, Y [B,T,2] f32, M [B,T] bool (scored rows; all required rows if no mask)."""
        B = len(ids)
        X = np.empty((B, SEQUENCE_LENGTH, N_FEATURES), np.float32)
        Y = np.empty((B, SEQUENCE_LENGTH, 2), np.float32)
        M = np.empty((B, SEQUENCE_LENGTH), bool)
        list(self._pool.map(lambda ig: self._load_into(ig[0], ig[1], X, Y, M), enumerate(ids)))
        return X, Y, M

    def batches(self, ids, batch_size):
        """Yield loaded batches, prefetching the next one in the background."""
        chunks = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]
        if not chunks:
            return
        prefetch = ThreadPoolExecutor(1)
        fut = prefetch.submit(self.load, chunks[0])
        for k in range(len(chunks)):
            batch = fut.result()
            if k + 1 < len(chunks):
                fut = prefetch.submit(self.load, chunks[k + 1])
            yield batch
        prefetch.shutdown()


def split_train(n_seqs: int, n_holdout: int, seed: int = 0):
    """Deterministic split of train row groups into (train_ids, holdout_ids)."""
    perm = np.random.default_rng(seed).permutation(n_seqs)
    return np.sort(perm[n_holdout:]), np.sort(perm[:n_holdout])

Writing /content/wnn/pipeline/data.py


In [8]:
%%writefile /content/wnn/pipeline/model.py
"""Stateful GRU model, metric-aligned losses, ONNX step export and baseline import."""
import numpy as np
import torch
from torch import nn

N_FEATURES = 112
CLIP = 2.0


class GRUNet(nn.Module):
    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0):
        super().__init__()
        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout)
        self.use_diff = use_diff
        in_dim = N_FEATURES * (2 if use_diff else 1)
        self.gru = nn.GRU(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, 2)

    def forward(self, x, h=None, x_prev=None):
        """x [B,T,112]; h [L,B,H] or None; x_prev [B,112] = last row of the previous chunk
        (None at sequence start, where the first diff is zero)."""
        if self.use_diff:
            first = x[:, :1] if x_prev is None else x_prev[:, None]
            prev = torch.cat([first, x[:, :-1]], 1)
            x = torch.cat([x, x - prev], -1)
        out, h = self.gru(x, h)
        return self.head(out), h


def wmse_loss(p, y):
    """Weighted MSE with the metric's clipping and |y| weights. p, y: [N, 2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()


def wpearson(p, y):
    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> [2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    W = w.sum(0) + 1e-8
    yc = y - (w * y).sum(0) / W
    pc = p - (w * p).sum(0) / W
    cov = (w * yc * pc).sum(0)
    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)


def compute_loss(kind, p, y, alpha=0.1):
    if kind == "wmse":
        return wmse_loss(p, y)
    if kind == "pearson":
        return 1 - wpearson(p, y).mean()
    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping
        return 1 - wpearson(p, y).mean() + alpha * wmse_loss(p, y)
    raise ValueError(kind)


class _Step(nn.Module):
    """One-row step graph for ONNX: (x [1,1,112], x_prev [1,112], h [L,1,H]) -> (pred [1,1,2], h)."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x, x_prev, h):
        pred, h = self.net(x, h, x_prev if self.net.use_diff else None)
        return pred, h


def export_onnx(net: GRUNet, path: str):
    net = net.eval().cpu()
    c = net.config
    args = (torch.zeros(1, 1, N_FEATURES), torch.zeros(1, N_FEATURES), torch.zeros(c["layers"], 1, c["hidden"]))
    with torch.no_grad():
        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", "h"],
                          output_names=["pred", "h_out"], opset_version=17, dynamo=False)


def load_baseline(onnx_path: str) -> GRUNet:
    """Load the organisers' 2-layer GRU ONNX weights into a GRUNet (for checks / warm starts)."""
    import onnx
    from onnx import numpy_helper

    g = onnx.load(onnx_path).graph
    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}
    grus = [n for n in g.node if n.op_type == "GRU"]
    matmul = next(n for n in g.node if n.op_type == "MatMul")
    H = 128
    net = GRUNet(hidden=H, layers=len(grus))

    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n
        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)

    sd = {}
    for l, n in enumerate(grus):
        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))
        sd[f"gru.weight_ih_l{l}"] = reorder(W)
        sd[f"gru.weight_hh_l{l}"] = reorder(R)
        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])
        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])
    sd["head.weight"] = init[matmul.input[1]].T
    sd["head.bias"] = init["reg_head.bias"]
    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in sd.items()})
    return net

Writing /content/wnn/pipeline/model.py


In [9]:
%%writefile /content/wnn/pipeline/fast_eval.py
"""Batched evaluation: runs whole sequences through the network at once (seconds instead of
the ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.

    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052
    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500
"""
import argparse
import json
import time

import numpy as np
import torch

from utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator

from .data import SeqStore, split_train
from .model import GRUNet, load_baseline

NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def predict_batch(net, X, device, chunk=2000):
    B, T, _ = X.shape
    out = np.empty((B, T, 2), np.float32)
    h = xp = None
    for s in range(0, T, chunk):
        x = torch.from_numpy(X[:, s:s + chunk]).to(device)
        p, h = net(x, h, xp)
        xp = x[:, -1]
        out[:, s:s + chunk] = p.float().cpu().numpy()
    return out


def evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):
    """Score several models in one pass over the data. Per model:
    {'scored': WP on scored rows, 'all': WP on all required rows, ...} (equal without a mask)."""
    for net in nets:
        net.eval()
    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]
    for X, Y, M in store.batches(list(ids), batch_size):
        for net, (acc, acc_all) in zip(nets, accs):
            P = predict_batch(net, X, device)
            for b in range(len(X)):
                acc.add(Y[b], P[b], M[b])
                if store.has_mask:
                    acc_all.add(Y[b], P[b], NEED)
    out = []
    for acc, acc_all in accs:
        r = acc.result()
        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": r["blocks"]}
        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else res["scored"]
        out.append(res)
    return out


def evaluate(net, store: SeqStore, ids, device, batch_size=128):
    return evaluate_many([net], store, ids, device, batch_size)[0]


def load_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = GRUNet(**ck["config"])
    net.load_state_dict(ck["state_dict"])
    return net


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--baseline", action="store_true")
    g.add_argument("--ckpt")
    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")
    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train holdout")
    ap.add_argument("--device", default="auto")
    args = ap.parse_args()

    device = pick_device(args.device)
    net = (load_baseline("baseline/baseline.onnx") if args.baseline else load_ckpt(args.ckpt)).to(device)
    out = {}
    t = time.time()
    valid = SeqStore("datasets/valid.parquet")
    out["valid"] = evaluate(net, valid, range(args.valid_seqs or valid.n_seqs), device)
    if args.holdout:
        train = SeqStore("datasets/train.parquet")
        _, hold = split_train(train.n_seqs, args.holdout)
        out["holdout"] = evaluate(net, train, hold, device)
    out["seconds"] = round(time.time() - t, 1)
    print(json.dumps(out, indent=1))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/fast_eval.py


In [10]:
%%writefile /content/wnn/pipeline/trainer.py
"""Shared training loop: one data stream feeds any number of experiments.

Every batch of whole sequences is read from parquet once and then used to train each model in
turn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= 99).
Evaluation is shared the same way: one pass over the eval data scores all models.
"""
import json
import math
import os
import shutil
import time
from dataclasses import asdict, dataclass, field

import numpy as np
import torch

from utils import WARMUP

from .data import SeqStore, split_train
from .fast_eval import evaluate_many
from .model import GRUNet, compute_loss, export_onnx, load_baseline

HERE = os.path.dirname(os.path.abspath(__file__))


@dataclass
class ExpConfig:
    """Per-model settings. Everything about the data stream is shared (see Common)."""
    name: str
    hidden: int = 128
    layers: int = 2
    diff: bool = False
    dropout: float = 0.0
    init_baseline: bool = False
    loss: str = "wmse"          # wmse | pearson | mix
    alpha: float = 0.1          # wmse weight in mix
    lr: float = 2e-3
    wd: float = 1e-4
    seed: int = 0


@dataclass
class Common:
    epochs: int = 4
    batch: int = 256            # sequences per batch
    chunk: int = 500            # TBPTT length
    clip_grad: float = 1.0
    train_seqs: int | None = None
    holdout: int = 500
    valid_seqs: int = 400
    evals_per_epoch: int = 2
    seed: int = 0               # data order + holdout split
    device: str = "auto"
    threads: int = 4            # parquet reader threads
    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)
    cache_train: bool = False   # keep training sequences in RAM after first read
    data_dir: str = "datasets"
    out_dir: str = "runs"
    extra: dict = field(default_factory=dict)


class Experiment:
    def __init__(self, cfg: ExpConfig, common: Common, device, total_steps):
        self.cfg = cfg
        torch.manual_seed(cfg.seed)
        if cfg.init_baseline:
            assert not cfg.diff and cfg.hidden == 128 and cfg.layers == 2, "baseline shape is fixed"
            self.net = load_baseline("baseline/baseline.onnx")
        else:
            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout)
        self.net.to(device)
        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, total_steps=total_steps,
                                                         pct_start=0.05)
        self.dir = os.path.join(common.out_dir, cfg.name)
        os.makedirs(self.dir, exist_ok=True)
        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")
        self.best = -1.0
        self.params = sum(p.numel() for p in self.net.parameters())

    def log(self, **kw):
        line = json.dumps(kw)
        self.log_f.write(line + "\n")
        self.log_f.flush()
        print(f"[{self.cfg.name}] {line}", flush=True)


def _store(common, name):
    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), n_threads=common.threads)


def train_many(configs: list[ExpConfig], common: Common):
    from .fast_eval import pick_device

    device = pick_device(common.device)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    rng = np.random.default_rng(common.seed)
    train, valid = _store(common, "train"), _store(common, "valid")
    if common.cache_train:
        train.enable_cache()
    if common.cache_eval:
        valid.enable_cache()
    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)
    if common.train_seqs:
        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)
    valid_ids = list(range(min(common.valid_seqs, valid.n_seqs)))
    hold_store = train
    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of train
        hold_store = _store(common, "train")
        hold_store.enable_cache()

    n_batches = math.ceil(len(train_ids) / common.batch)
    total_steps = common.epochs * n_batches * math.ceil(20000 / common.chunk)
    exps = [Experiment(c, common, device, total_steps) for c in configs]
    t0 = time.time()
    for e in exps:
        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), params=e.params,
              train_seqs=len(train_ids), holdout=len(hold_ids), steps=total_steps)
    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)

    def run_eval(epoch, bi):
        nets = [e.net for e in exps]
        vs = evaluate_many(nets, valid, valid_ids, device)
        hs = evaluate_many(nets, hold_store, hold_ids, device)
        for e, v, h in zip(exps, vs, hs):
            sel = (v["scored"] + h["all"]) / 2  # don't select on the visible valid mask alone
            is_best = sel > e.best
            if is_best:
                e.best = sel
                torch.save({"config": e.net.config, "state_dict": e.net.state_dict(), "exp": asdict(e.cfg)},
                           os.path.join(e.dir, "best.pt"))
            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), valid_all=round(v["all"], 6),
                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, time=round(time.time() - t0, 1))
            e.net.train()

    for epoch in range(common.epochs):
        for e in exps:
            e.net.train()
        ids = list(rng.permutation(train_ids))
        for bi, (X, Y, _) in enumerate(train.batches(ids, common.batch)):
            tb = time.time()
            Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy per batch
            Yd = torch.from_numpy(Y).to(device, non_blocking=True)
            for e in exps:
                h = xp = None
                losses = []
                for s in range(0, X.shape[1], common.chunk):
                    x, y = Xd[:, s:s + common.chunk], Yd[:, s:s + common.chunk]
                    p, h = e.net(x, h, xp)
                    lo = max(0, WARMUP - s)  # skip warm-up rows
                    if lo < x.shape[1]:
                        loss = compute_loss(e.cfg.loss, p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2),
                                            e.cfg.alpha)
                        e.opt.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)
                        e.opt.step()
                        e.sched.step()
                        losses.append(loss.item())
                    h = h.detach()
                    xp = x[:, -1]
                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(np.mean(losses)), 5),
                      lr=round(e.sched.get_last_lr()[0], 6))
            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{common.epochs}: {time.time() - tb:.1f}s "
                  f"for {len(exps)} models, elapsed {(time.time() - t0) / 60:.1f} min", flush=True)
            del Xd, Yd
            if bi in eval_at:
                run_eval(epoch, bi)

    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary table
    for e in exps:
        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", weights_only=False)
        e.net.load_state_dict(ck["state_dict"])
        e.net.to(device)
    nets = [e.net for e in exps]
    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # uncached: 17 GB
    full_hold = evaluate_many(nets, hold_store, hold_ids, device)
    rows = []
    for e, v, h in zip(exps, full_valid, full_hold):
        sub = os.path.join(e.dir, "submission")
        os.makedirs(sub, exist_ok=True)
        export_onnx(e.net, os.path.join(sub, "model.onnx"))
        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, "solution.py"))
        res = {"valid_full": v, "holdout": h}
        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)
        e.log(event="done", **res)
        rows.append({"name": e.cfg.name, "params": e.params, "valid_scored": round(v["scored"], 6),
                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),
                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4)})
    rows.sort(key=lambda r: -(r["valid_scored"] + r["holdout_all"]))
    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("\n" + " | ".join(rows[0].keys()))
    for r in rows:
        print(" | ".join(str(v) for v in r.values()))
    return rows

Writing /content/wnn/pipeline/trainer.py


In [11]:
%%writefile /content/wnn/pipeline/multi_train.py
"""Train several experiments side by side from one data stream.

    python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 --cache-eval

experiments.json is a list of ExpConfig dicts, e.g.
    [{"name": "wmse"}, {"name": "mix", "loss": "mix"}, {"name": "diff_h192", "diff": true, "hidden": 192}]
Shared settings (epochs, batch, chunk, data/out dirs, ...) are the Common fields, as flags.
"""
import argparse
import dataclasses
import json

from .trainer import Common, ExpConfig, train_many


def add_common_flags(ap):
    for f in dataclasses.fields(Common):
        if f.name == "extra":
            continue
        flag = "--" + f.name.replace("_", "-")
        if f.type is bool:
            ap.add_argument(flag, action="store_true")
        else:  # train_seqs defaults to None (= all) but takes an int
            ap.add_argument(flag, type=int if f.default is None else type(f.default), default=f.default)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True, help="JSON list of experiment configs")
    add_common_flags(ap)
    args = vars(ap.parse_args())
    configs = [ExpConfig(**c) for c in json.load(open(args.pop("config")))]
    names = [c.name for c in configs]
    assert len(set(names)) == len(names), "experiment names must be unique"
    train_many(configs, Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/multi_train.py


In [12]:
%%writefile /content/wnn/pipeline/train.py
"""Train one stateful GRU (thin wrapper over trainer.train_many; see multi_train.py for several).

    python -m pipeline.train --name quick --train-seqs 1000 --epochs 2
    python -m pipeline.train --name gru128_wmse --epochs 4

Outputs in runs/NAME/: best.pt, log.jsonl, results.json, submission/{solution.py, model.onnx}.
"""
import argparse
import dataclasses

from .multi_train import add_common_flags
from .trainer import Common, ExpConfig, train_many


def main():
    ap = argparse.ArgumentParser()
    exp_fields = {f.name: f for f in dataclasses.fields(ExpConfig)}
    for name, f in exp_fields.items():
        flag = "--" + name.replace("_", "-")
        if name == "name":
            ap.add_argument(flag, required=True)
        elif f.type is bool:
            ap.add_argument(flag, action="store_true")
        else:
            ap.add_argument(flag, type=type(f.default), default=f.default)
    add_common_flags(ap)
    args = vars(ap.parse_args())
    exp = ExpConfig(**{k: args.pop(k) for k in list(args) if k in exp_fields})
    train_many([exp], Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/train.py


In [13]:
%%writefile /content/wnn/pipeline/solution_template.py
"""Generic stateful ONNX step runner. Copied as solution.py next to model.onnx by pipeline.train."""
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import onnxruntime as ort


class PredictionModel:
    def __init__(self):
        o = ort.SessionOptions()
        o.intra_op_num_threads = 1
        o.inter_op_num_threads = 1
        o.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        o.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        o.add_session_config_entry("session.intra_op.allow_spinning", "0")
        self.session = ort.InferenceSession(str(Path(__file__).resolve().parent / "model.onnx"),
                                            sess_options=o, providers=["CPUExecutionProvider"])
        shapes = {i.name: i.shape for i in self.session.get_inputs()}
        self.h0 = np.zeros(shapes["h"], dtype=np.float32)
        self.use_prev = "x_prev" in shapes  # the exporter drops x_prev when the model has no diff features
        self.seq_ix = None

    def predict(self, data_point):
        if data_point.seq_ix != self.seq_ix:
            self.seq_ix = data_point.seq_ix
            self.h = self.h0.copy()
            self.x_prev = None
        x = np.nan_to_num(np.asarray(data_point.state, dtype=np.float32)).reshape(1, 1, 112)
        feeds = {"x": x, "h": self.h}
        if self.use_prev:
            feeds["x_prev"] = x[0] if self.x_prev is None else self.x_prev
        pred, self.h = self.session.run(None, feeds)
        self.x_prev = x[0]
        if not data_point.need_prediction:
            return None
        return np.nan_to_num(pred.reshape(2).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

Writing /content/wnn/pipeline/solution_template.py


## 3. Sanity check: baseline weights through our fast evaluator (full valid gives 0.617052)

In [14]:
!python -m pipeline.fast_eval --baseline --valid-seqs 300

{
 "valid": {
  "scored": 0.6246286498889674,
  "t0": 0.6038379686442771,
  "t1": 0.6454193311336577,
  "n_seqs": 300,
  "all": 0.4380707773677913
 },
 "seconds": 4.9
}


## 4. Experiments
Edit freely. Per-model fields: `name, hidden, layers, diff, dropout, init_baseline, loss (wmse|pearson|mix),
alpha, lr, wd, seed`. Shared settings are flags on the run command below.

Inference budget: the competition allows ~91 µs/row on 1 CPU; the 128-unit baseline takes ~22 µs on an M4 Mac.
Hidden 192 costs roughly 2x, hidden 256 roughly 3.5x: check with `evaluate.py` locally before submitting.

In [16]:
import json
experiments = [
 {
  "name": "e01_wmse_h128"
 },
 {
  "name": "e02_mix_h128",
  "loss": "mix",
  "alpha": 0.1
 },
 {
  "name": "e03_pearson_h128",
  "loss": "pearson"
 },
 {
  "name": "e04_wmse_diff_h128",
  "diff": True
 },
 {
  "name": "e05_wmse_h192_drop",
  "hidden": 192,
  "dropout": 0.1
 },
 {
  "name": "e06_wmse_h128_lr1e-3",
  "lr": 0.001
 }
]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')

6 experiments


## 5. Train
Quick smoke test first (uncomment), then the real run. Per-batch logs are in `OUT_DIR/<name>/log.jsonl`;
only evals and timing are printed here.

In [17]:
# !python -m pipeline.multi_train --config experiments.json --epochs 1 --train-seqs 256 --holdout 100 --valid-seqs 100 --threads 8 --out-dir /content/smoke
!python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 --cache-eval --out-dir {OUT_DIR} 2>&1 | grep -v '"event": "batch"'

[e01_wmse_h128] {"event": "start", "config": {"name": "e01_wmse_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 0}, "common": {"epochs": 4, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_per_epoch": 2, "seed": 0, "device": "auto", "threads": 8, "cache_eval": true, "cache_train": false, "data_dir": "datasets", "out_dir": "/content/drive/MyDrive/wunder-connectome/runs", "extra": {}}, "device": "cuda", "params": 192258, "train_seqs": 10107, "holdout": 500, "steps": 6400}
[e02_mix_h128] {"event": "start", "config": {"name": "e02_mix_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "mix", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 0}, "common": {"epochs": 4, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_per_epo

## 6. Results

In [18]:
import pandas as pd
rows = [json.loads(l) for l in open(f'{OUT_DIR}/summary.jsonl')]
df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
df['select'] = (df.valid_scored + df.holdout_all) / 2
df.sort_values('select', ascending=False)

,name,params,valid_scored,valid_all,holdout_all,t0,t1,select
0,e05_wmse_h192_drop,398978,0.677301,0.506302,0.451641,0.6730,0.6816,0.564471
1,e04_wmse_diff_h128,235266,0.677120,0.503275,0.451014,0.6728,0.6814,0.564067
2,e02_mix_h128,192258,0.677836,0.506474,0.450050,0.6733,0.6824,0.563943
3,e01_wmse_h128,192258,0.675942,0.504676,0.447134,0.6712,0.6807,0.561538
4,e06_wmse_h128_lr1e-3,192258,0.676342,0.504269,0.444559,0.6714,0.6813,0.560450
5,e03_pearson_h128,192258,0.656165,0.475497,0.418525,0.6512,0.6611,0.537345


Each `OUT_DIR/<name>/submission/` holds `solution.py` + `model.onnx`. Copy the best ones to the Mac and check speed with `evaluate.py` before uploading.